# Imports and predictions

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

from demand.config import PROCESSED
from demand.metrics import evaluate

predictions = pd.read_parquet(PROCESSED / "predictions.parquet")
cv = predictions[predictions.fold <= 8]
predictions.groupby("model").size()

model
lgbm_scaled           6250
lstm                  6250
ridge                 6250
rnn                   6250
seasonal_median_4w    6250
xgb_scaled            6250
dtype: int64

# Metrics by model

In [5]:
by_fold = cv.groupby(["model", "fold"]).apply(
    lambda d: pd.Series(evaluate(d.sales, d.pred)), include_groups=False
)
summary = by_fold.groupby("model").mean().sort_values("WAPE")
summary["WAPE gain"] = (1 - summary.WAPE / summary.loc["seasonal_median_4w", "WAPE"]).round(3)
summary.round(3)

,MAE,RMSE,WAPE,WAPE gain
model,,,,
xgb_scaled,517.253,883.648,0.105,0.254
lgbm_scaled,527.891,889.865,0.107,0.238
rnn,611.088,1058.315,0.125,0.114
lstm,616.568,1030.884,0.125,0.108
seasonal_median_4w,695.353,1193.748,0.141,0.000
ridge,746.590,1140.484,0.152,-0.078


# Significance vs baseline

In [6]:
units = {
    model: group.assign(err=(group.sales - group.pred).abs())
    .groupby(["fold", "store_nbr", "family"]).agg(err=("err", "sum"), actual=("sales", "sum"))
    for model, group in cv.groupby("model")
}
wape = by_fold["WAPE"].unstack("model")


def compare(reference, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for model in wape.columns.drop(reference):
        ref_u, mod_u = units[reference], units[model]
        assert ref_u.index.equals(mod_u.index)
        pick = rng.integers(0, len(ref_u), size=(n_boot, len(ref_u)))
        actual = ref_u.actual.to_numpy()[pick].sum(axis=1)
        gain = (ref_u.err.to_numpy()[pick].sum(axis=1) - mod_u.err.to_numpy()[pick].sum(axis=1)) / actual
        low, high = np.percentile(gain, [2.5, 97.5])
        rows.append({
            "model": model,
            "WAPE gain": wape[reference].mean() - wape[model].mean(),
            "95% CI low": low,
            "95% CI high": high,
            "folds better": int((wape[model] < wape[reference]).sum()),
            "Wilcoxon p": wilcoxon(wape[model], wape[reference], alternative="less").pvalue,
        })
    return pd.DataFrame(rows).set_index("model").round(4)


compare("seasonal_median_4w")

,WAPE gain,95% CI low,95% CI high,folds better,Wilcoxon p
model,,,,,
lgbm_scaled,0.0335,0.0256,0.0434,8,0.0039
lstm,0.0152,0.0060,0.0263,3,0.4727
ridge,-0.0110,-0.0181,-0.0033,4,0.8438
rnn,0.0161,0.0089,0.0251,6,0.0742
xgb_scaled,0.0357,0.0276,0.0454,8,0.0039


# Significance vs SGBoost (scaled)

In [7]:
compare("xgb_scaled")

,WAPE gain,95% CI low,95% CI high,folds better,Wilcoxon p
model,,,,,
lgbm_scaled,-0.0022,-0.0041,-0.0004,2,0.9609
lstm,-0.0205,-0.0251,-0.0156,0,1.0000
ridge,-0.0467,-0.0548,-0.0399,0,1.0000
rnn,-0.0196,-0.0254,-0.0141,2,0.9805
seasonal_median_4w,-0.0357,-0.0454,-0.0276,0,1.0000


# Hold-out check

In [8]:
hold = predictions[predictions.fold == 9]
holdout_summary = (
    hold.groupby("model")
    .apply(lambda d: pd.Series(evaluate(d.sales, d.pred)), include_groups=False)
    .sort_values("WAPE")
)
holdout_summary["WAPE gain"] = (
    1 - holdout_summary.WAPE / holdout_summary.loc["seasonal_median_4w", "WAPE"]
).round(3)
holdout_summary["WAPE in CV"] = summary.WAPE
holdout_summary.round(3)

,MAE,RMSE,WAPE,WAPE gain,WAPE in CV
model,,,,,
xgb_scaled,405.363,679.886,0.091,0.230,0.105
lgbm_scaled,411.964,684.178,0.093,0.218,0.107
rnn,459.644,700.974,0.103,0.127,0.125
lstm,461.784,692.396,0.104,0.123,0.125
seasonal_median_4w,526.541,899.182,0.118,0.000,0.141
ridge,599.561,879.023,0.135,-0.139,0.152


# Conclusions

- Setup: six models are compared on the same 8 walk-forward folds (28 days each, open days only) and once on the 28-day hold-out (2017-07-19 to 2017-08-15), which was not used for any choice.
- Ranking in walk-forward (WAPE): XGBoost with scaled target 0.105, LightGBM 0.107, RNN 0.125, LSTM 0.125, seasonal median baseline 0.141, Ridge 0.152.
- Against the baseline, XGBoost and LightGBM are better in 8 of 8 folds (Wilcoxon p 0.004, bootstrap intervals above zero), with a gain of 0.036 and 0.034 WAPE (25% and 24%).
- RNN (better in 6 of 8 folds, p 0.07) and LSTM (3 of 8 folds, p 0.47) are not convincingly better than the baseline; the LSTM gain comes mostly from one fold. Ridge is significantly worse than the baseline.
- No model is better than XGBoost. LightGBM is practically tied (0.002 WAPE), the networks are about 0.02 worse, Ridge 0.047 worse.
- Hold-out: XGBoost with scaled target reaches WAPE 0.091 (MAE 405, RMSE 680), 23% below the baseline (0.118). The order of the models is the same as in walk-forward. All errors, including the baseline, are lower than in walk-forward, so this month is easier than the average fold.
- Winner: XGBoost with the target divided by the series level. The networks need about 50 times more training time and give a worse result.
- Limits: the bootstrap over (fold, series) units ignores the correlation between consecutive folds, so its intervals are narrow; the fold-level test has only 8 pairs; the hold-out is one 28-day window (700 rows); features, scaling and hyperparameters were chosen on the same walk-forward folds.